# 🏆 Capstone Step 2 — Intelligence Core (M3 + M5)
**Prerequisite:** Complete `lesson_09a_ingestion_layer.ipynb` first

In [ ]:
# ── Capstone imports & env check ──────────────────────────────────
import os, json, time, datetime, re, asyncio
from pathlib import Path
from typing import Optional, List, Any
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field as dc_field
import pandas as pd
import numpy as np

from openai import OpenAI
from anthropic import Anthropic
from datasets import load_dataset
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Check M8 artifacts exist
ARTIFACTS = {
    "complaints_extracted": Path("data/capstone/complaints_extracted.jsonl"),
    "hybrid_routing_log":   Path("data/capstone/hybrid_routing_log.jsonl"),
    "golden_eval":          Path("data/capstone/golden_eval.jsonl"),
}
for name, path in ARTIFACTS.items():
    status = "✅" if path.exists() else "⚠️  MISSING — run Module 8 first"
    print(f"  {status}  {name}: {path}")

print("\n✅ Capstone imports ready")

---
## Step 2 — Intelligence Core (M3 + M5)
**Goal:** Wire ChromaDB policy store + XGBoost churn model into `hybrid_route()`.

**Auto-check verifies:**
- ✓ ChromaDB with ≥ 30 policies indexed
- ✓ XGBoost AUC ≥ 0.68
- ✓ `hybrid_routing_log.jsonl` saved


In [ ]:
show_exercise(
    "CAP-2", "Intelligence Core",
    "Build ChromaDB with ≥ 30 synthetic policy docs. Train XGBoost on Telco Churn. "
    "Implement hybrid_route(): ML for high-confidence, LLM+RAG for uncertain.",
    hints=[
        "▸ Synthetic policies: lm_client.chat.completions.create(...) for each topic",
        "▸ hybrid_route() logs: {customer_id, ml_proba, route, final_prediction, cost_usd}",
        "▸ Reuse ChromaDB setup and XGBoost model from Module 8",
    ],
    checks=[
        "ChromaDB with ≥ 30 policies indexed",
        "XGBoost AUC ≥ 0.68",
        "hybrid_route() log saved to hybrid_routing_log.jsonl"
    ]
)

In [ ]:
# ── Step 2: Load/rebuild Intelligence Core ────────────────────────
import chromadb, xgboost as xgb
from chromadb.utils import embedding_functions
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# ─ Policy ChromaDB ────────────────────────────────────────────────
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client  = chromadb.Client()
try:
    chroma_client.delete_collection("cap_policies")
except Exception:
    pass
policy_collection = chroma_client.create_collection("cap_policies", embedding_function=ef)

POLICY_TOPICS = [
    "credit card dispute", "mortgage fees", "overdraft protection", "fraud reporting",
    "loan repayment", "savings interest", "wire transfer", "KYC verification",
    "GDPR rights", "AML checks", "account closure", "foreign transaction fees",
    "direct debit rules", "standing orders", "chargeback process",
]
policy_docs_cap = []
for topic in POLICY_TOPICS:
    for v in range(2):   # 30 = 15 × 2
        msg = lm_client.chat.completions.create(
            model="local-model", max_tokens=180,
            messages=[{"role":"user","content":f"Write a 120-word formal BANK POLICY on: {topic} (v{v+1})"}]
        )
        policy_docs_cap.append({"id":f"p{len(policy_docs_cap):03d}","text":msg.choices[0].message.content,"topic":topic})

policy_collection.add(
    ids       =[d["id"]   for d in policy_docs_cap],
    documents =[d["text"] for d in policy_docs_cap],
    metadatas =[{"topic": d["topic"]} for d in policy_docs_cap],
)
print(f"✅ ChromaDB: {policy_collection.count()} policies indexed")

def retrieve_cap_policy(query: str, n: int = 2) -> str:
    results = policy_collection.query(query_texts=[query], n_results=n)
    return "\n---\n".join(results["documents"][0])

In [ ]:
# ─ XGBoost churn model ────────────────────────────────────────────
try:
    telco = pd.read_csv("data/telco_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
except FileNotFoundError:
    np.random.seed(42); n=700
    telco = pd.DataFrame({
        "tenure":np.random.randint(1,72,n),"MonthlyCharges":np.random.uniform(20,120,n),
        "TotalCharges":np.random.uniform(100,8000,n),
        "Contract":np.random.choice(["Month-to-month","One year","Two year"],n),
        "PaymentMethod":np.random.choice(["Electronic check","Mailed check","Bank transfer"],n),
        "InternetService":np.random.choice(["DSL","Fiber optic","No"],n),
        "Churn":np.random.choice(["Yes","No"],n,p=[0.27,0.73]),
    })
    print("⚠️  Using synthetic Telco data")

telco = telco.dropna()
telco["Churn_bin"] = (telco["Churn"]=="Yes").astype(int)
NUM_COLS = ["tenure","MonthlyCharges","TotalCharges"]
CAT_COLS = [c for c in ["Contract","PaymentMethod","InternetService"] if c in telco.columns]

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS)
])
churn_model = Pipeline([
    ("pre", pre),
    ("clf", xgb.XGBClassifier(n_estimators=150, max_depth=4, random_state=42,
                               eval_metric="logloss", verbosity=0))
])
X = telco[NUM_COLS+CAT_COLS]; y = telco["Churn_bin"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
churn_model.fit(Xtr, ytr)
AUC = roc_auc_score(yte, churn_model.predict_proba(Xte)[:,1])
print(f"✅ XGBoost AUC = {AUC:.3f}")

In [ ]:
# ─ hybrid_route() ─────────────────────────────────────────────────
CAP_HYBRID_LOG = []

def get_churn_risk(customer_features: dict) -> ChurnRiskCard:
    defaults = {"tenure":12,"MonthlyCharges":65.0,"TotalCharges":800.0,
                "Contract":"Month-to-month","PaymentMethod":"Electronic check","InternetService":"DSL"}
    row = {**defaults, **customer_features}
    X_row = pd.DataFrame([{k: row[k] for k in NUM_COLS+CAT_COLS}])
    proba = float(churn_model.predict_proba(X_row)[0,1])
    tier  = "high" if proba > 0.65 else ("medium" if proba > 0.4 else "low")
    return ChurnRiskCard(
        customer_id=str(customer_features.get("customer_id","unknown")),
        churn_proba=round(proba,3), risk_tier=tier,
        top_features=["tenure","MonthlyCharges","Contract"],
    )

def hybrid_route_cap(customer_features: dict, query: str) -> dict:
    risk = get_churn_risk(customer_features)
    proba = risk.churn_proba
    if proba > 0.75 or proba < 0.25:
        route = "ml"
        cost  = 0.0
    else:
        retrieve_cap_policy(query)
        route = "llm_rag"
        cost  = 0.0002
    entry = {"customer_id":customer_features.get("customer_id","?"),
             "ml_proba":proba,"route":route,
             "final_prediction":risk.risk_tier,
             "cost_usd":cost,"ts":datetime.datetime.now().isoformat(timespec="seconds")}
    CAP_HYBRID_LOG.append(entry)
    return entry

# Run on 10 test customers
for i in range(10):
    hybrid_route_cap({"customer_id":f"C{i:03d}","tenure":i*6,"MonthlyCharges":40+i*8},
                     "churn risk assessment")

log_path = Path("data/capstone/hybrid_routing_log.jsonl")
with open(log_path,"w") as f:
    for e in CAP_HYBRID_LOG: f.write(json.dumps(e)+"\n")
print(f"✅ Routing log: {len(CAP_HYBRID_LOG)} entries → {log_path}")

check([
    (policy_collection.count() >= 30, f"≥ 30 policies ({policy_collection.count()} indexed)"),
    (AUC >= 0.68,                     f"XGBoost AUC ≥ 0.68 (got {AUC:.3f})"),
    (len(CAP_HYBRID_LOG) >= 5,        "Hybrid router ran on ≥ 5 customers"),
    (log_path.exists(),               "hybrid_routing_log.jsonl saved"),
], exercise_id="CAP-2")

---
## ✅ Step 2 Complete
- ☐ ChromaDB with ≥ 30 policies indexed
- ☐ XGBoost AUC ≥ 0.68
- ☐ `hybrid_routing_log.jsonl` saved

➡️ Continue to `lesson_09c_agent_orchestrator.ipynb`